![Itq](img/LogoITQ.jpg)

**Propueta para la implementación de Fast Api para uso de neurona MP**

![Python](img/python_logo.png)

* *Nombre:* Judith Piedra
* *Fecha :* **13-10-2026**
* *Repositorio:* <a href= "https://github.com/MerakiDevJP/machine2_23092026nocturno.git"> Link del repositorio Github</a>
* https://github.com/MerakiDevJP/machine2_23092026nocturno.git

# Plan de desarrollo Fast Api

**Fase 1**
**Análisis y Selección de Características (Feature Selection)**

El objetivo de esta fase es determinar las 5 características de mayor poder diagnóstico para utilizar en la Fast Api.

1. Importar el dataset de Cáncer de Mama de Wisconsin desde scikit-learn y estructurarlo en un DataFrame de Pandas.

2. Aplicar un modelo auxiliar (como Random Forest o un análisis de correlación de Pearson) para identificar qué variables aportan la mayor ganancia de información.

3. Seleccionar formalmente las 5 características críticas basadas en patología tumoral y dispersión estadística, y saber si coinciden con las variables investigadas: worst radius, worst perimeter, worst concave points, worst area, mean concave points

Justificación de selección:
- La etiqueta worst representan el promedio de los tres valores más altos encontrados en las células analizadas de la muestra. En un tumor maligno, siempre existen áreas de crecimiento agresivo o "nódulos satélites" donde las dimensiones celulares alcanzan sus valores más extremos. Estas variaciones extremas son el indicador más claro de malignidad.
- worst radius (Radio peor) y worst area (Área peor), los tumores malignos ocupan mucho más espacio y crecen con mayor volumen. El radio y el área máxima reflejan directamente la expansión tridimensional de la lesión en el tejido mamario.
- worst perimeter (Perímetro peor), el contorno de una célula o grupo celular maligno suele ser rugoso, anguloso y deforme debido a la rápida división celular. Un perímetro máximo elevado combinado con un área grande delata una estructura celular altamente desorganizada.
- worst concave points y mean concave points (Puntos cóncavos), los "puntos cóncavos" miden las indentaciones o porciones hacia adentro en el contorno de la forma nuclear o celular. Las células sanas o benignas tienden a tener formas ovaladas o circulares regulares (convexas). Por el contrario, los núcleos de las células cancerosas presentan pliegues, muescas y deformaciones profundas (múltiples puntos cóncavos). Estadísticamente, en la literatura médica y de minería de datos de este dataset, los concave points son históricamente las variables con mayor correlación lineal directa frente al diagnóstico de malignidad.



In [1]:
# ==========================================
# 1. CARGA Y EXPLORACIÓN INICIAL
# ==========================================
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier

# Cargar el dataset de Cáncer de Mama de Wisconsin
cancer_mama = load_breast_cancer()

# Estructurarlo en un DataFrame de Pandas con los nombres reales de las características
df_full = pd.DataFrame(cancer_mama.data, columns=cancer_mama.feature_names)
Y = cancer_mama.target

print(f"Dimensiones del dataset completo: {df_full.shape}")
print("Primeras características del dataset:")
display(df_full.head(3))




Dimensiones del dataset completo: (569, 30)
Primeras características del dataset:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758


In [2]:
# ==========================================
# 2. EVALUACIÓN DE IMPORTANCIA (RANDOM FOREST)
# ==========================================
# Entrenamos un modelo de Bosques Aleatorios para medir la ganancia de información (Feature Importance)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(df_full, Y)

# Crear un DataFrame con la importancia de cada variable
feature_importances = pd.DataFrame({
    'Característica': df_full.columns,
    'Importancia': rf_model.feature_importances_
}).sort_values(by='Importancia', ascending=False)

print("\n--- Ranking de Importancia de Variables (Top 10) ---")
display(feature_importances.head(10))




--- Ranking de Importancia de Variables (Top 10) ---


,Característica,Importancia
23,worst area,0.139357
27,worst concave points,0.132225
7,mean concave points,0.107046
20,worst radius,0.082848
22,worst perimeter,0.080850
2,mean perimeter,0.067990
6,mean concavity,0.066917
3,mean area,0.060462
26,worst concavity,0.037339
0,mean radius,0.034843


In [3]:

# ==========================================
# 3. FILTRADO DE LAS 5 VARIABLES CLAVE
# ==========================================
# Se seleccionan las 5 características críticas basadas en patología tumoral y dispersión estadística
top_5_features = [
    'worst area',
    'worst concave points',
    'mean concave points',
    'worst radius', 
    'worst perimeter'    
]
# DataFrame reducido para analizar si luego se usa en la Neurona MP
df_reducido = df_full[top_5_features]

print("\n--- DataFrame Reducido con las 5 Características Clave ---")
display(df_reducido.head(5))


--- DataFrame Reducido con las 5 Características Clave ---


,worst area,worst concave points,mean concave points,worst radius,worst perimeter
0,2019.0,0.2654,0.14710,25.38,184.60
1,1956.0,0.1860,0.07017,24.99,158.80
2,1709.0,0.2430,0.12790,23.57,152.50
3,567.7,0.2575,0.10520,14.91,98.87
4,1575.0,0.1625,0.10430,22.54,152.20


**Fase 2**
**Preprocesamiento y Entrenamiento de la Neurona MP**

Neurona procesada en el documento 11.CasoPractivoNeuronaMP.ipynb, se incluyen al final la selección de las carácterísticas que se realizó en la Fase 1, se entrena y se guarda el archivo para el uso posterior. 

**Fase 3**
**Construcción del Backend (FastAPI)**

Crear la pasarela lógica que mantendrá el modelo entrenado en memoria y procesará las peticiones HTTP del usuario.

- Configuración del entorno: Instalar las librerías necesarias (fastapi, uvicorn, pandas, numpy, scikit-learn).

- Estructura del servidor (main.py):
    * Utilizar el modelo al iniciar la aplicación para que esté listo para predecir.
    * Definir un esquema de validación con Pydantic que asegure la recepción exacta de las 5 variables requeridas.
      
- Creación de Endpoints:
    * Endpoint de la interfaz (/): Retornará el código HTML del formulario.
    * Endpoint de predicción (/): Recibirá los datos del paciente, los binarizará, ejecutará el método .predict() de la neurona y devolverá el diagnóstico ("Benigno" o "Maligno").



In [4]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import numpy as np
import pandas as pd
import pickle
import threading

# 1. Aplicar parche para entornos asíncronos en Jupyter
nest_asyncio.apply()

# ==========================================
# 2. CARGAR EL MODELO DESDE EL .PKL
# ==========================================
class MpNeurona:
    def __init__(self):
        self.threshold = None
    def model(self, x):
        z = sum(x)
        return (z >= self.threshold)
    def predict(self, X):
        Y_preds = []
        for x in X:
            result = self.model(x)
            Y_preds.append(result)
        return np.array(Y_preds)

print("Iniciando servidor: Cargando el modelo entrenado (.pkl)...")
with open('modelo_neurona_mp.pkl', 'rb') as archivo:
    datos_cargados = pickle.load(archivo)

np_neurona_cargada = datos_cargados['neurona']
top_features = datos_cargados['top_features']
print(f"¡Modelo cargado con éxito! Umbral óptimo en memoria: {np_neurona_cargada.threshold}")
print(f"Características asociadas: {top_features}")

# ==========================================
# 3. ESTRUCTURA DEL SERVIDOR Y CORS
# ==========================================
app = FastAPI(title="API Neurona MP - Cáncer de Mama")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class PatientData(BaseModel):
    worst_radius: float
    worst_perimeter: float
    worst_concave_points: float
    worst_area: float
    mean_concave_points: float

# ==========================================
# 4. ENDPOINT DE LA INTERFAZ (GET /) - FASE 4
# ==========================================
@app.get("/", response_class=HTMLResponse)
def serve_frontend():
    return """
    <!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Diagnóstico de Cáncer de Mama - Neurona MP</title>
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    </head>
    <body class="bg-light">
        <div class="container mt-5 mb-5">
            <div class="row justify-content-center">
                <div class="col-md-7">
                    <div class="card shadow-lg border-0 rounded-4">
                        <div class="card-header bg-dark text-white text-center py-4 rounded-top-4">
                            <h3 class="mb-0 fw-bold">Evaluador Clínico Inteligente</h3>
                            <p class="text-muted text-light small mb-0 mt-1">Modelo de Neurona de McCulloch-Pitts (5 Variables Clave)</p>
                        </div>
                        <div class="card-body p-4 p-md-5">
                            <form id="predictionForm">
                                <div class="mb-3">
                                    <label for="worst_radius" class="form-label fw-semibold">Worst Radius (Radio Máximo):</label>
                                    <input type="number" step="any" class="form-control" id="worst_radius" placeholder="Ej: 14.5" required>
                                </div>
                                <div class="mb-3">
                                    <label for="worst_perimeter" class="form-label fw-semibold">Worst Perimeter (Perímetro Máximo):</label>
                                    <input type="number" step="any" class="form-control" id="worst_perimeter" placeholder="Ej: 95.0" required>
                                </div>
                                <div class="mb-3">
                                    <label for="worst_concave_points" class="form-label fw-semibold">Worst Concave Points (Puntos Cóncavos Máx.):</label>
                                    <input type="number" step="any" class="form-control" id="worst_concave_points" placeholder="Ej: 0.08" required>
                                </div>
                                <div class="mb-3">
                                    <label for="worst_area" class="form-label fw-semibold">Worst Area (Área Máxima):</label>
                                    <input type="number" step="any" class="form-control" id="worst_area" placeholder="Ej: 650.0" required>
                                </div>
                                <div class="mb-4">
                                    <label for="mean_concave_points" class="form-label fw-semibold">Mean Concave Points (Media Puntos Cóncavos):</label>
                                    <input type="number" step="any" class="form-control" id="mean_concave_points" placeholder="Ej: 0.05" required>
                                </div>
                                <button type="submit" class="btn btn-primary w-100 py-2 fw-bold shadow-sm">Consultar Diagnóstico</button>
                            </form>
                            <div id="resultado" class="mt-4 text-center" style="display: none;"></div>
                        </div>
                    </div>
                </div>
            </div>
        </div>
        <script>
            document.getElementById('predictionForm').addEventListener('submit', async function(e) {
                e.preventDefault();
                
                const data = {
                    worst_radius: parseFloat(document.getElementById('worst_radius').value),
                    worst_perimeter: parseFloat(document.getElementById('worst_perimeter').value),
                    worst_concave_points: parseFloat(document.getElementById('worst_concave_points').value),
                    worst_area: parseFloat(document.getElementById('worst_area').value),
                    mean_concave_points: parseFloat(document.getElementById('mean_concave_points').value)
                };

                const resDiv = document.getElementById('resultado');
                resDiv.style.display = 'block';
                resDiv.className = "alert alert-info";
                resDiv.innerHTML = "Analizando datos con el modelo...";

                try {
                    const response = await fetch('/predict', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify(data)
                    });

                    if (!response.ok) {
                        const errorText = await response.text();
                        throw new Error(`Error del servidor (${response.status}): ${errorText}`);
                    }

                    const result = await response.json();
                    
                    if (result.prediction === "Benigno") {
                        resDiv.className = "alert alert-success p-3 shadow-sm text-start";
                        resDiv.innerHTML = `
                            <h5 class="fw-bold mb-1 text-center">Resultado: ${result.prediction}</h5>
                            <hr>
                            <p class="small mb-0 text-muted fst-italic"><strong>Notificación técnica:</strong> ${result.observacion_tecnica}</p>
                        `;
                    } else {
                        resDiv.className = "alert alert-danger p-3 shadow-sm text-start";
                        resDiv.innerHTML = `
                            <h5 class="fw-bold mb-1 text-center">Resultado: ${result.prediction}</h5>
                            <hr>
                            <p class="small mb-0 text-muted fst-italic"><strong>Notificación técnica:</strong> ${result.observacion_tecnica}</p>
                        `;
                    }
                } catch (error) {
                    console.error("Detalle del error:", error);
                    resDiv.className = "alert alert-danger";
                    resDiv.innerHTML = "<strong>Error:</strong> " + error.message;
                }
            });
        </script>
    </body>
    </html>
    """

# ==========================================
# 5. ENDPOINT DE PREDICCIÓN (POST /predict)
# ==========================================
@app.post("/predict")
def predict_diagnosis(patient: PatientData):
    try:
        valores = [
            patient.worst_radius,
            patient.worst_perimeter,
            patient.worst_concave_points,
            patient.worst_area,
            patient.mean_concave_points
        ]
        
        # Llaves corregidas con espacios para coincidir exactamente con top_features del .pkl
        puntos_medios = {
            'worst radius': 21.95,        
            'worst perimeter': 150.0,      
            'worst concave points': 0.145, 
            'worst area': 2219.5,         
            'mean concave points': 0.10   
        }
        
        binarios = []
        for i, key in enumerate(top_features):
            val = valores[i]
            limite = puntos_medios[key]
            bin_val = 1 if val <= limite else 0
            binarios.append(bin_val)
            
        vector_binario = np.array(binarios)
        
        pred_array = np_neurona_cargada.predict(vector_binario.reshape(1, -1))
        pred = bool(pred_array[0])
        
        diagnosis = "Benigno" if pred else "Maligno"
        
        return {
            "prediction": diagnosis, 
            "raw_output": pred,
            "vector_evaluado": vector_binario.tolist(),
            "observacion_tecnica": (
                "Se aplicó una binarización mediante puntos medios de referencia "
                "debido a la restricción matemática del modelo de Neurona de McCulloch-Pitts (1943), "
                "la cual opera estrictamente con entradas binarias (0 y 1) y umbrales fijos sin ponderación sináptica."
            )
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# ==========================================
# 6. LANZAR EL SERVIDOR EN SEGUNDO PLANO
# ==========================================
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("¡Servidor FastAPI corriendo con éxito en segundo plano!")
print("Abre tu navegador en: http://127.0.0.1:8000")

Iniciando servidor: Cargando el modelo entrenado (.pkl)...
¡Modelo cargado con éxito! Umbral óptimo en memoria: 5
Características asociadas: ['worst radius', 'worst perimeter', 'worst concave points', 'worst area', 'mean concave points']
¡Servidor FastAPI corriendo con éxito en segundo plano!
Abre tu navegador en: http://127.0.0.1:8000


**Fase 4**
**Desarrollo de la Interfaz de Usuario (Frontend HTML/Bootstrap)**

Construir una interfaz limpia, ligera y fácil de usar que evite la fricción de ingresar 30 parámetros clínicos.
- Diseño del Formulario: Crear un panel central utilizando Bootstrap con 5 campos de entrada numéricos (input type="number"), uno para cada característica seleccionada.
- Lógica de Conexión (JavaScript):
    * Capturar el evento de envío (submit) del formulario.
    * Empaquetar los valores en formato JSON.
    * Enviar una petición asíncrona mediante fetch() hacia la ruta /predict de la API de FastAPI.
- Visualización de Resultados: Mostrar dinámicamente una alerta visual en pantalla indicando el resultado del diagnóstico de forma inmediata.



In [5]:
# ==========================================
# FASE 4: DESARROLLO DE LA INTERFAZ DE USUARIO (FRONTEND)
# ==========================================

@app.get("/", response_class=HTMLResponse)
def serve_frontend():
    return """
    <!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Diagnóstico de Cáncer de Mama - Neurona MP</title>
        <!-- Importar Bootstrap 5 CSS para un diseño limpio, ligero y profesional -->
        <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    </head>
    <body class="bg-light">

        <div class="container mt-5 mb-5">
            <div class="row justify-content-center">
                <div class="col-md-7">
                    
                    <!-- Tarjeta Principal del Formulario -->
                    <div class="card shadow-lg border-0 rounded-4">
                        <div class="card-header bg-dark text-white text-center py-4 rounded-top-4">
                            <h3 class="mb-0 fw-bold">Evaluador Clínico Inteligente</h3>
                            <p class="text-muted text-light small mb-0 mt-1">Modelo de Neurona de McCulloch-Pitts (5 Variables Clave)</p>
                        </div>
                        
                        <div class="card-body p-4 p-md-5">
                            <!-- Formulario con los 5 campos numéricos estrictos -->
                            <form id="predictionForm">
                                
                                <div class="mb-3">
                                    <label for="worst_radius" class="form-label fw-semibold">Worst Radius (Radio Máximo):</label>
                                    <input type="number" step="any" class="form-control" id="worst_radius" placeholder="Ej: 18.25" required>
                                </div>

                                <div class="mb-3">
                                    <label for="worst_perimeter" class="form-label fw-semibold">Worst Perimeter (Perímetro Máximo):</label>
                                    <input type="number" step="any" class="form-control" id="worst_perimeter" placeholder="Ej: 122.8" required>
                                </div>

                                <div class="mb-3">
                                    <label for="worst_concave_points" class="form-label fw-semibold">Worst Concave Points (Puntos Cóncavos Máx.):</label>
                                    <input type="number" step="any" class="form-control" id="worst_concave_points" placeholder="Ej: 0.141" required>
                                </div>

                                <div class="mb-3">
                                    <label for="worst_area" class="form-label fw-semibold">Worst Area (Área Máxima):</label>
                                    <input type="number" step="any" class="form-control" id="worst_area" placeholder="Ej: 1040.0" required>
                                </div>

                                <div class="mb-4">
                                    <label for="mean_concave_points" class="form-label fw-semibold">Mean Concave Points (Media Puntos Cóncavos):</label>
                                    <input type="number" step="any" class="form-control" id="mean_concave_points" placeholder="Ej: 0.085" required>
                                </div>

                                <button type="submit" class="btn btn-primary w-100 py-2 fw-bold shadow-sm">Consultar Diagnóstico</button>
                            </form>
                            
                            <!-- Contenedor dinámico para mostrar la alerta visual de resultados -->
                            <div id="resultado" class="mt-4 text-center" style="display: none;"></div>
                        </div>
                    </div>
                    
                </div>
            </div>
        </div>

        <!-- Lógica de Conexión y Asincronía con JavaScript (Fetch API) -->
        <script>
            document.getElementById('predictionForm').addEventListener('submit', async function(e) {
                e.preventDefault();
                
                // 1. Capturar y empaquetar los valores en formato JSON
                const data = {
                    worst_radius: parseFloat(document.getElementById('worst_radius').value),
                    worst_perimeter: parseFloat(document.getElementById('worst_perimeter').value),
                    worst_concave_points: parseFloat(document.getElementById('worst_concave_points').value),
                    worst_area: parseFloat(document.getElementById('worst_area').value),
                    mean_concave_points: parseFloat(document.getElementById('mean_concave_points').value)
                };

                const resDiv = document.getElementById('resultado');
                resDiv.style.display = 'block';
                resDiv.className = "alert alert-info";
                resDiv.innerHTML = "Analizando datos con el modelo...";

                try {
                    // 2. Enviar petición asíncrona mediante fetch() hacia la ruta /predict
                    const response = await fetch('/predict', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify(data)
                    });

                    if (!response.ok) {
                        throw new Error("Error en la comunicación con el servidor.");
                    }

                    const result = await response.json();
                    
                    // 3. Visualización dinámica del resultado y la notificación técnica de binarización
                    if (result.prediction === "Benigno") {
                        resDiv.className = "alert alert-success p-3 shadow-sm text-start";
                        resDiv.innerHTML = `
                            <h5 class="fw-bold mb-1 text-center">Resultado: ${result.prediction}</h5>
                            <hr>
                            <p class="small mb-0 text-muted fst-italic"><strong>Notificación técnica:</strong> ${result.observacion_tecnica}</p>
                        `;
                    } else {
                        resDiv.className = "alert alert-danger p-3 shadow-sm text-start";
                        resDiv.innerHTML = `
                            <h5 class="fw-bold mb-1 text-center">Resultado: ${result.prediction}</h5>
                            <hr>
                            <p class="small mb-0 text-muted fst-italic"><strong>Notificación técnica:</strong> ${result.observacion_tecnica}</p>
                        `;
                    }
                } catch (error) {
                    resDiv.className = "alert alert-danger";
                    resDiv.innerHTML = "Ocurrió un error al procesar la solicitud con la API.";
                }
            });
        </script>
    </body>
    </html>
    """

In [6]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Cargar el dataset original
cancer = load_breast_cancer()
df_full = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# Seleccionar las 5 características de tu proyecto
top_features = [
    'worst radius', 
    'worst perimeter', 
    'worst concave points', 
    'worst area', 
    'mean concave points'
]

df = df_full[top_features]

# Construir tabla con Mínimo, Máximo y Punto Medio (Límite de binarización)
resumen_limites = pd.DataFrame({
    'Mínimo': df.min(),
    'Máximo': df.max(),
    'Punto Medio (Corte Binarización)': (df.min() + df.max()) / 2
})

print(resumen_limites)

                      Mínimo     Máximo  Punto Medio (Corte Binarización)
worst radius            7.93    36.0400                           21.9850
worst perimeter        50.41   251.2000                          150.8050
worst concave points    0.00     0.2910                            0.1455
worst area            185.20  4254.0000                         2219.6000
mean concave points     0.00     0.2012                            0.1006


**Fase 5**
**Pruebas, Validación y Despliegue**

Garantizar el correcto funcionamiento del sistema completo.


In [ ]:
import uvicorn

# Configuramos Uvicorn deshabilitando los websockets para evitar el conflicto de versiones
config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, ws="none", log_level="info")
server = uvicorn.Server(config)

print("¡Iniciando servidor FastAPI en Jupyter...")
print("Haz clic en el siguiente enlace para abrir la interfaz web: http://127.0.0.1:8000")

# Ejecutar el servidor de forma asíncrona compatible con el entorno interactivo
await server.serve()

INFO:     Started server process [9760]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
Exception in thread Thread-3 (run_server):
Traceback (most recent call last):
  File "C:\Users\JUDY\anaconda3\envs\machinelearning2026nocturno\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\JUDY\anaconda3\envs\machinelearning2026nocturno\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\JUDY\AppData\Local\Temp\ipykernel_9760\1917612528.py", line 224, in run_server
  File "C:\Users\JUDY\anaconda3\envs\machinelearning2026nocturno\Lib\site-packages\uvicorn\main.py", line 632, in run
    server.run()
  File "C:\Users\JUDY\anaconda3\envs\machinelearning2026nocturno\Lib\site-packages\uvicorn\server.py", line 86, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
 

¡Iniciando servidor FastAPI en Jupyter...
Haz clic en el siguiente enlace para abrir la interfaz web: http://127.0.0.1:8000
